In [18]:
import os
import re
import json
from datasets import Dataset

def clean_output_tags(text):
    """
    终极版标签修复：暴力拆解后重新组装，免疫所有嵌套、重复、乱序问题
    """
    if not text:
        return text

    # 1. 剔除所有现存的 <think> 和 </think> 标签
    text = re.sub(r'</?think>', '', text, flags=re.IGNORECASE)
    # 2. 剔除所有毫无意义的空 <final></final> 标签
    text = re.sub(r'<final>\s*</final>', '', text, flags=re.IGNORECASE)
    text = text.strip()
    
    # 3. 重新规范化组装
    if re.search(r'<final>', text, flags=re.IGNORECASE):
        text = re.sub(r'<final>', '</think>\n<final>', text, count=1, flags=re.IGNORECASE)
        text = f"<think>\n{text}"
    else:
        text = f"<think>\n{text}\n</think>"
        
    return text

def process_grpo_data(input_file, output_dir, train_filename, val_filename, test_size=0.1):
    os.makedirs(output_dir, exist_ok=True)
    processed_records = []
    
    print(f"正在读取并清洗数据: {input_file}")
    with open(input_file, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            
            data = json.loads(line)
            
            # 1. 拼接 prompt
            instruction = data.get("instruction", "").strip()
            input_text = data.get("input", "").strip()
            user_content = instruction + ("\n" + input_text if input_text else "")
            prompt = [{"role": "user", "content": user_content}]
            
            # 2. 清洗 output 标签
            raw_output = data.get("output", "")
            cleaned_output = clean_output_tags(raw_output)
            
            # 3. 提取 ground_truth：严格提取 <final>...</final> 之间的完整内容，不做任何简化！
            match = re.search(r"<final>(.*?)</final>", cleaned_output, re.DOTALL)
            ground_truth = match.group(1).strip() if match else cleaned_output.strip()
            
            # 4. 组装行数据
            record = {
                "prompt": prompt,
                "ground_truth": ground_truth,          # 外层：完整答案
                "reward_model": {
                    "ground_truth": ground_truth     # reward_model层：完全一致，不简化
                },
                "raw_output": cleaned_output,          # 修复好标签的完整输出
                "difficulty": data.get("difficulty", ""),
                "view": data.get("view", "")
            }
            processed_records.append(record)

    # 转化为 Dataset 并划分
    dataset = Dataset.from_list(processed_records)
    print(f"正在划分数据集 (验证集比例: {test_size})...")
    split_dataset = dataset.train_test_split(test_size=test_size, seed=42)
    
    train_out_path = os.path.join(output_dir, train_filename)
    val_out_path = os.path.join(output_dir, val_filename)
    
    split_dataset['train'].to_parquet(train_out_path)
    split_dataset['test'].to_parquet(val_out_path)
    
    print(f"✅ 处理完成！")
    print(f"📦 训练集 ({len(split_dataset['train'])} 条) -> {train_out_path}")
    print(f"📦 验证集 ({len(split_dataset['test'])} 条) -> {val_out_path}")

if __name__ == "__main__":
    # 配置路径
    INPUT_JSONL = '/mnt/data/zwl/verl/data/mixed_40.jsonl'
    OUTPUT_DIR = '/mnt/data/zwl/verl/data/rl'
    TRAIN_FILE = 'qwen3_4b_grpo_mixed_train.parquet'
    VAL_FILE = 'qwen3_4b_grpo_mixed_val.parquet'
    
    process_grpo_data(
        input_file=INPUT_JSONL,
        output_dir=OUTPUT_DIR,
        train_filename=TRAIN_FILE,
        val_filename=VAL_FILE,
        test_size=0.1
    )

正在读取并清洗数据: /mnt/data/zwl/verl/data/mixed_40.jsonl
正在划分数据集 (验证集比例: 0.1)...


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 1868.29ba/s]

✅ 处理完成！
📦 训练集 (36 条) -> /mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_train.parquet
📦 验证集 (4 条) -> /mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_val.parquet


In [19]:
import pyarrow.parquet as pq

# 读取 Parquet 文件
table = pq.read_table('/mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_train.parquet')

# 转换为 pandas DataFrame
df = table.to_pandas()

print(f"行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")

行数: 36
列名: ['prompt', 'ground_truth', 'reward_model', 'raw_output', 'difficulty', 'view']


In [20]:
import pyarrow.parquet as pq
import pandas as pd
import json

# 读取 Parquet 文件
table = pq.read_table('/mnt/data/zwl/verl/data/rl/qwen3_4b_grpo_mixed_train.parquet')
df = table.to_pandas()

print(f"行数: {len(df)}")
print(f"列名: {df.columns.tolist()}")
print(f"数据形状: {df.shape}")
print("\n" + "="*80)
print("前 1 条数据（完整内容）:")
print("="*80)

# 设置 pandas 显示选项，显示完整内容
pd.set_option('display.max_colwidth', None)  # 不限制列宽
pd.set_option('display.max_rows', None)       # 不限制行数
pd.set_option('display.width', None)          # 不限制宽度
pd.set_option('display.max_seq_items', None)  # 不限制序列项

# 方法1: 直接显示
print(df.head(1))

print("\n" + "="*80)
print("逐列详细查看:")
print("="*80)

# 方法2: 逐列显示完整内容
for col in df.columns:
    print(f"\n列名: {col}")
    print(f"数据类型: {df[col].dtype}")
    print(f"前1条内容:")
    try:
        # 尝试美化显示
        value = df[col].iloc[0]
        if isinstance(value, (dict, list)):
            print(json.dumps(value, ensure_ascii=False, indent=2))
        else:
            print(value)
    except Exception as e:
        print(f"无法显示: {e}")

print("\n" + "="*80)
print("数据统计信息:")
print("="*80)
print(df.describe())

# 如果有字符串列，显示长度信息
print("\n" + "="*80)
print("字符串列长度信息:")
print("="*80)
for col in df.select_dtypes(include=['object']).columns:
    try:
        print(f"{col}: 最小长度={df[col].str.len().min()}, 最大长度={df[col].str.len().max()}, 平均长度={df[col].str.len().mean():.2f}")
    except:
        pass

行数: 36
列名: ['prompt', 'ground_truth', 'reward_model', 'raw_output', 'difficulty', 'view']
数据形状: (36, 6)

前 1 条数据（完整内容）:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                

/tmp/ipykernel_15404/3188049470.py:53: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=['object']).columns:
